# Mission 3 P0: 'predict at least 1 symptom' post-processing

Reuses the P0 script's own functions (importlib from file path) to rebuild the identical 80:20 multilabel-stratified split (seed 42), loads the re-run checkpoint `mission3_tfidf_logreg.joblib`, and evaluates: if no label has p >= 0.5, set the argmax label to 1. Script is not modified.

In [1]:
# Cell 0: config + import script module
import os, sys, platform, hashlib, subprocess, json, time, importlib.util
from pathlib import Path
import numpy as np, pandas as pd, joblib, sklearn
from sklearn.metrics import f1_score, precision_recall_fscore_support
os.environ["PYTHONIOENCODING"] = "utf-8"

REPO = Path(r"C:\Users\<user>\Desktop\아주대\dcc\01_prelim\dcc2026-prelim")
LABEL_DIR = REPO / "data" / "raw_merged" / "Training" / "2.라벨링데이터" / "TL_서울_구급"
SCRIPT = REPO / "scripts" / "mission3_baseline 최종 수정본.py"
OUT = REPO / "outputs" / "m3_p0_rerun"
CKPT = OUT / "mission3_tfidf_logreg.joblib"
SEED, TEST_SIZE, TH = 42, 0.2, 0.5

def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

HEAD = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("git HEAD     :", HEAD)
print("script sha256:", sha256(SCRIPT))
print("ckpt sha256  :", sha256(CKPT))
print("python", sys.version.split()[0], "| sklearn", sklearn.__version__, "|", platform.platform())

spec = importlib.util.spec_from_file_location("m3_baseline", SCRIPT)
m3 = importlib.util.module_from_spec(spec)
sys.modules["m3_baseline"] = m3
spec.loader.exec_module(m3)
assert m3.MultilabelStratifiedShuffleSplit is not None, "iterstrat not importable"
LABELS = list(m3.TARGET_SYMPTOMS)
print("TARGET_SYMPTOMS:", LABELS)


git HEAD     : 35c607584d48b758aa9a4de67a6e7624ad62f7b3
script sha256: fee2bef3d52568fd8ecfb31773f0b8987a3dc4a3517508d858fdac9fc7209103
ckpt sha256  : d9b9df04f90784b499b5b2cc2eb55d630d899d44bbe35e72fdeaceaa0c30beb9
python 3.11.9 | sklearn 1.8.0 | Windows-10-10.0.26200-SP0


TARGET_SYMPTOMS: ['고열', '구토', '두통', '복통', '어지러움', '열상', '오심', '전신쇠약', '호흡곤란']


In [2]:
# Cell 1: load data + rebuild the identical split with the script's own functions
import random
from sklearn.preprocessing import MultiLabelBinarizer
random.seed(SEED); np.random.seed(SEED)   # same as train_baseline()
t0 = time.time()
df = m3.load_training_dataframe(LABEL_DIR, None)
summary = m3.make_dataset_summary(df)
print(f"loaded {len(df):,} rows in {time.time()-t0:.1f}s | load failures: {summary['load_failure_count']}")
assert len(df) == 29200 and summary["load_failure_count"] == 0

mlb = MultiLabelBinarizer(classes=m3.TARGET_SYMPTOMS)
mlb.fit([m3.TARGET_SYMPTOMS])
train_df, valid_df = m3._make_split(df=df, test_size=TEST_SIZE, seed=SEED, mlb=mlb)
y_train = mlb.transform(train_df["symptoms"])
y_valid = mlb.transform(valid_df["symptoms"])
print(f"train {len(train_df):,} | val {len(valid_df):,}")
assert len(valid_df) == 5792 and len(train_df) == 23408

# cross-check against the rerun's validation_predictions.csv (same file order)
vp = pd.read_csv(OUT / "validation_predictions.csv", encoding="utf-8-sig")
assert vp["label file name"].tolist() == valid_df["label_file_name"].tolist(), "val file order differs from rerun"
print("val file list/order identical to rerun validation_predictions.csv")


loaded 29,200 rows in 998.6s | load failures: 0


train 23,408 | val 5,792
val file list/order identical to rerun validation_predictions.csv


In [3]:
# Cell 2: zero-positive-label counts (target 9 symptoms)
zero_all = df["symptoms"].map(len).eq(0)
zero_tr = train_df["symptoms"].map(len).eq(0)
zero_va = valid_df["symptoms"].map(len).eq(0)
print(f"all  : {int(zero_all.sum()):,} / {len(df):,} calls have 0 target labels ({zero_all.mean()*100:.2f}%)")
print(f"train: {int(zero_tr.sum()):,} / {len(train_df):,} ({zero_tr.mean()*100:.2f}%)")
print(f"val  : {int(zero_va.sum()):,} / {len(valid_df):,} ({zero_va.mean()*100:.2f}%)")
print("script make_dataset_summary zero_target_label_count:", summary["zero_target_label_count"])
nt = df.loc[zero_all, "non_target_symptoms"].map(len).gt(0)
print(f"of the zero-target calls: {int(nt.sum()):,} have only non-target symptoms, {int((~nt).sum()):,} have an empty symptom field")
print("empty-text calls:", summary["empty_text_count"])
print("\nlabel-cardinality distribution (all):")
print(df["symptoms"].map(len).value_counts().sort_index().to_string())
if zero_all.any():
    from collections import Counter
    c = Counter(s for l in df.loc[zero_all, "non_target_symptoms"] for s in l)
    print("\ntop non-target symptoms among zero-target calls:", c.most_common(10))


all  : 0 / 29,200 calls have 0 target labels (0.00%)
train: 0 / 23,408 (0.00%)
val  : 0 / 5,792 (0.00%)
script make_dataset_summary zero_target_label_count: 0
of the zero-target calls: 0 have only non-target symptoms, 0 have an empty symptom field
empty-text calls: 0

label-cardinality distribution (all):
symptoms
1    19560
2     7137
3     2161
4      325
5       17


In [4]:
# Cell 3: load rerun checkpoint, baseline check vs metrics.json
ck = joblib.load(CKPT)
assert list(ck["target_symptoms"]) == LABELS and float(ck["threshold"]) == TH
vec, clf = ck["vectorizer"], ck["classifier"]

# features sanity: refit the script's vectorizer config on train and compare vocab
from sklearn.feature_extraction.text import TfidfVectorizer
v2 = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
v2.fit(train_df["text"])
assert v2.vocabulary_ == vec.vocabulary_, "refit vocabulary differs from checkpoint vectorizer"
print("checkpoint vectorizer vocabulary == refit on rebuilt train split:", len(vec.vocabulary_), "terms")

X_valid = vec.transform(valid_df["text"])
P = clf.predict_proba(X_valid)
pred_base = (P >= TH).astype(np.int8)

def scores(y, p):
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, average=None, zero_division=0)
    out = {
        "macro_f1": float(f1_score(y, p, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(y, p, average="micro", zero_division=0)),
    }
    for avg in ("macro", "micro"):
        a, b, _, _ = precision_recall_fscore_support(y, p, average=avg, zero_division=0)
        out[f"precision_{avg}"], out[f"recall_{avg}"] = float(a), float(b)
    per = pd.DataFrame({"symptom": LABELS, "support": sup.astype(int), "precision": pr, "recall": rc, "f1": f1,
                        "n_pred_pos": p.sum(axis=0).astype(int)})
    return out, per

base, per_base = scores(y_valid, pred_base)
ref = json.loads((OUT / "metrics.json").read_text(encoding="utf-8"))
print(f"baseline macro_f1 = {base['macro_f1']:.8f} | metrics.json = {ref['macro_f1']:.8f} | |diff| = {abs(base['macro_f1']-ref['macro_f1']):.2e}")
assert abs(base["macro_f1"] - ref["macro_f1"]) <= 1e-6
assert abs(base["micro_f1"] - ref["micro_f1"]) <= 1e-6
print("baseline reproduced within 1e-6")
print(per_base.to_string(index=False))


checkpoint vectorizer vocabulary == refit on rebuilt train split: 50000 terms


baseline macro_f1 = 0.56755207 | metrics.json = 0.56755207 | |diff| = 0.00e+00
baseline reproduced within 1e-6
symptom  support  precision   recall       f1  n_pred_pos
     고열     1037   0.567869 0.637416 0.600636        1164
     구토      893   0.454280 0.522956 0.486205        1028
     두통      581   0.397083 0.421687 0.409015         617
     복통     1354   0.783333 0.694239 0.736100        1200
   어지러움     1267   0.577334 0.639305 0.606742        1403
     열상      693   0.732779 0.890332 0.803909         842
     오심      668   0.258604 0.393713 0.312166        1017
   전신쇠약     1062   0.443787 0.635593 0.522648        1521
   호흡곤란      785   0.575789 0.696815 0.630548         950


In [5]:
# Cell 4: min-1 post-processing
no_pos = pred_base.sum(axis=1) == 0
pred_min1 = pred_base.copy()
am = P.argmax(axis=1)
pred_min1[np.where(no_pos)[0], am[no_pos]] = 1
n_aff = int(no_pos.sum())
hit = y_valid[np.where(no_pos)[0], am[no_pos]] == 1
true_empty_aff = int((y_valid[no_pos].sum(axis=1) == 0).sum())
print(f"val rows with no label >= {TH}: {n_aff:,} / {len(valid_df):,} ({n_aff/len(valid_df)*100:.2f}%)")
print(f"  forced argmax label correct (TP): {int(hit.sum()):,} | wrong (FP): {int((~hit).sum()):,} | of affected rows with 0 true labels: {true_empty_aff:,}")
print(f"  max prob among affected rows: median {np.median(P[no_pos].max(axis=1)) if n_aff else float('nan'):.3f}")
forced = pd.DataFrame({"symptom": LABELS,
                       "forced_n": [int((am[no_pos] == j).sum()) for j in range(len(LABELS))],
                       "forced_tp": [int(((am[no_pos] == j) & hit).sum()) for j in range(len(LABELS))]})
forced["forced_fp"] = forced["forced_n"] - forced["forced_tp"]

after, per_after = scores(y_valid, pred_min1)
print(f"\nMacro F1: {base['macro_f1']:.5f} -> {after['macro_f1']:.5f} ({after['macro_f1']-base['macro_f1']:+.5f})")
print(f"Micro F1: {base['micro_f1']:.5f} -> {after['micro_f1']:.5f} ({after['micro_f1']-base['micro_f1']:+.5f})")
for k in ("precision_macro", "recall_macro", "precision_micro", "recall_micro"):
    print(f"{k}: {base[k]:.5f} -> {after[k]:.5f} ({after[k]-base[k]:+.5f})")

tab = per_base[["symptom", "support"]].copy()
for m in ("precision", "recall", "f1", "n_pred_pos"):
    tab[f"{m}_before"] = per_base[m]
    tab[f"{m}_after"] = per_after[m]
    if m != "n_pred_pos":
        tab[f"{m}_delta"] = per_after[m] - per_base[m]
tab = tab.merge(forced, on="symptom")
pd.set_option("display.width", 250)
print("\nper-label before/after:")
print(tab.round(4).to_string(index=False))
print("\nfocus:")
print(tab[tab.symptom.isin(["오심", "두통"])][["symptom", "f1_before", "f1_after", "f1_delta", "precision_before", "precision_after", "recall_before", "recall_after", "forced_n", "forced_tp"]].round(4).to_string(index=False))


val rows with no label >= 0.5: 293 / 5,792 (5.06%)
  forced argmax label correct (TP): 88 | wrong (FP): 205 | of affected rows with 0 true labels: 0
  max prob among affected rows: median 0.467

Macro F1: 0.56755 -> 0.56798 (+0.00043)
Micro F1: 0.57792 -> 0.57829 (+0.00036)
precision_macro: 0.53232 -> 0.52526 (-0.00706)
recall_macro: 0.61467 -> 0.62507 (+0.01039)
precision_micro: 0.53634 -> 0.52945 (-0.00689)
recall_micro: 0.62650 -> 0.63705 (+0.01055)

per-label before/after:
symptom  support  precision_before  precision_after  precision_delta  recall_before  recall_after  recall_delta  f1_before  f1_after  f1_delta  n_pred_pos_before  n_pred_pos_after  forced_n  forced_tp  forced_fp
     고열     1037            0.5679           0.5633          -0.0045         0.6374        0.6519        0.0145     0.6006    0.6044    0.0037               1164              1200        36         15         21
     구토      893            0.4543           0.4501          -0.0041         0.5230        0.5

In [6]:
# Cell 5: save outputs
res = {
    "rule": "if no label prob >= 0.5, set argmax label to 1",
    "threshold": TH, "seed": SEED, "test_size": TEST_SIZE,
    "repo_commit": HEAD, "ckpt": str(CKPT), "ckpt_sha256": sha256(CKPT), "script_sha256": sha256(SCRIPT),
    "n_total": int(len(df)), "n_train": int(len(train_df)), "n_val": int(len(valid_df)),
    "zero_label_all": int(zero_all.sum()), "zero_label_train": int(zero_tr.sum()), "zero_label_val": int(zero_va.sum()),
    "val_rows_affected": n_aff, "forced_tp": int(hit.sum()), "forced_fp": int((~hit).sum()),
    "affected_rows_with_zero_true_labels": true_empty_aff,
    "before": base, "after": after,
    "delta_macro_f1": after["macro_f1"] - base["macro_f1"],
    "delta_micro_f1": after["micro_f1"] - base["micro_f1"],
    "baseline_matches_metrics_json": True,
}
(OUT / "min1_metrics.json").write_text(json.dumps(res, ensure_ascii=False, indent=2), encoding="utf-8")
tab.to_csv(OUT / "min1_per_label.csv", index=False, encoding="utf-8-sig")
print(json.dumps(res, ensure_ascii=False, indent=2))
print("wrote", OUT / "min1_metrics.json", "and", OUT / "min1_per_label.csv")


{
  "rule": "if no label prob >= 0.5, set argmax label to 1",
  "threshold": 0.5,
  "seed": 42,
  "test_size": 0.2,
  "repo_commit": "35c607584d48b758aa9a4de67a6e7624ad62f7b3",
  "ckpt": "C:\\Users\\<user>\\Desktop\\아주대\\dcc\\01_prelim\\dcc2026-prelim\\outputs\\m3_p0_rerun\\mission3_tfidf_logreg.joblib",
  "ckpt_sha256": "d9b9df04f90784b499b5b2cc2eb55d630d899d44bbe35e72fdeaceaa0c30beb9",
  "script_sha256": "fee2bef3d52568fd8ecfb31773f0b8987a3dc4a3517508d858fdac9fc7209103",
  "n_total": 29200,
  "n_train": 23408,
  "n_val": 5792,
  "zero_label_all": 0,
  "zero_label_train": 0,
  "zero_label_val": 0,
  "val_rows_affected": 293,
  "forced_tp": 88,
  "forced_fp": 205,
  "affected_rows_with_zero_true_labels": 0,
  "before": {
    "macro_f1": 0.567552067774815,
    "micro_f1": 0.5779227961508683,
    "precision_macro": 0.5323176818092431,
    "recall_macro": 0.6146729338924944,
    "precision_micro": 0.5363375076986245,
    "recall_micro": 0.6264988009592326
  },
  "after": {
    "macro_f1":